**Linking to kaggle for dataset**

In [2]:
from google.colab import drive
from google.colab import files
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create project folders in your Drive
os.makedirs('/content/drive/MyDrive/fl-plant-disease/data', exist_ok=True)

# Upload your kaggle.json file
print("Please upload your kaggle.json file:")
uploaded = files.upload()

# Move the key to the correct hidden folder for Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API key configured successfully!")


Mounted at /content/drive
Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Kaggle API key configured successfully!


In [3]:
# Move to your Drive data folder
%cd /content/drive/MyDrive/fl-plant-disease/data/

# Download the dataset
!kaggle datasets download -d emmarex/plantdisease


/content/drive/MyDrive/fl-plant-disease/data
Dataset URL: https://www.kaggle.com/datasets/emmarex/plantdisease
License(s): unknown
100% 658M/658M [00:04<00:00, 168MB/s]



**At first run this to load the dataset in Colab local storage**

In [4]:
# 1. Copy the zip file from Drive to local Colab storage
!cp /content/drive/MyDrive/fl-plant-disease/data/plantdisease.zip /content/plantdisease.zip

# 2. Unzip quietly (-q) to the /content/dataset folder
!unzip -q /content/plantdisease.zip -d /content/dataset/

# 3. Remove the local zip file to save space
!rm /content/plantdisease.zip

# 4. Check if it worked!
!ls /content/dataset/


plantvillage  PlantVillage


In [5]:
# Check the contents of the PlantVillage folder
!ls /content/dataset/PlantVillage


Pepper__bell___Bacterial_spot  Tomato_Late_blight
Pepper__bell___healthy	       Tomato_Leaf_Mold
Potato___Early_blight	       Tomato_Septoria_leaf_spot
Potato___healthy	       Tomato_Spider_mites_Two_spotted_spider_mite
Potato___Late_blight	       Tomato__Target_Spot
Tomato_Bacterial_spot	       Tomato__Tomato_mosaic_virus
Tomato_Early_blight	       Tomato__Tomato_YellowLeaf__Curl_Virus
Tomato_healthy


In [6]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import os

# 1. Define the path to your images
# (Adjust this if Step 1 shows a different path, e.g., /content/dataset/plantvillage)
data_dir = '/content/dataset/PlantVillage'

# 2. Define Image Transformations
# Neural networks need images to be the same size and converted to Tensors.
# We use standard ImageNet normalization values.
transform = transforms.Compose([
    transforms.Resize((224, 224)),       # Standard size for MobileNet/EfficientNet
    transforms.ToTensor(),               # Convert image to PyTorch Tensor
    transforms.Normalize(                # Normalize pixel values
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# 3. Load the Dataset
# ImageFolder automatically assigns labels based on folder names
full_dataset = torchvision.datasets.ImageFolder(root=data_dir, transform=transform)
print(f"Total images found: {len(full_dataset)}")
print(f"Total classes found: {len(full_dataset.classes)}")

# 4. Split into Train (80%) and Validation (20%)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

# 5. Create DataLoaders
# DataLoaders handle batching, shuffling, and loading data in parallel
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("DataLoaders are ready!")


Total images found: 20638
Total classes found: 15
Training images: 16510
Validation images: 4128
DataLoaders are ready!


In [7]:
# Grab one batch of training data
images, labels = next(iter(train_loader))

print(f"Image batch shape: {images.shape}") # Expected: [32, 3, 224, 224] (Batch Size, Channels, Height, Width)
print(f"Label batch shape: {labels.shape}") # Expected: [32]


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
